# Unsupervised analysis — SmartSeq

Goal: discover structure in the SmartSeq data **without** using the
Hypo/Norm labels, then compare what we found with the labels to decide
whether hypoxia is the dominant transcriptional axis.

Pipeline (one per cell line):

1. Wrap the expression matrix as a `scanpy.AnnData` object.
2. Scale → PCA → nearest-neighbour graph → UMAP.
3. **Leiden clustering** at several resolutions; pick the resolution that
   maximises the **silhouette score** in PCA space.
4. **KMeans** for `k = 2..6` as a sanity check; pick the `k` with the
   best silhouette.
5. Compare the best clustering against the ground-truth condition with
   Adjusted Rand Index (ARI) and Normalised Mutual Information (NMI).

In [ ]:
import sys
from pathlib import Path

# Add project root so `from src.utils import ...` works regardless of where the notebook is run from.
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")
RNG = 42

In [ ]:
import scanpy as sc
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.cluster import KMeans

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")


def run_scanpy_pipeline(adata, n_pcs=30, n_neighbors=15, random_state=42):
    """Standard scanpy embedding pipeline.

    The teacher's data is already log-normalised on 3000 HVGs, so we skip
    `normalize_total` / `log1p` / `highly_variable_genes` and go straight to
    scaling + PCA.
    """
    sc.pp.scale(adata, max_value=10)
    sc.tl.pca(adata, n_comps=n_pcs, random_state=random_state)
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs, random_state=random_state)
    sc.tl.umap(adata, random_state=random_state)
    return adata


def silhouette_sweep_leiden(adata, resolutions, random_state=42):
    """Sweep Leiden resolution, scoring each clustering by silhouette in PCA space."""
    rows = []
    X_pca = adata.obsm["X_pca"]
    for res in resolutions:
        key = f"leiden_r{res:.2f}"
        sc.tl.leiden(
            adata,
            resolution=res,
            random_state=random_state,
            key_added=key,
            flavor="igraph",
            n_iterations=2,
            directed=False,
        )
        labels = adata.obs[key].astype(int).values
        n_clusters = len(np.unique(labels))
        sil = silhouette_score(X_pca, labels) if n_clusters > 1 else np.nan
        rows.append({"resolution": res, "n_clusters": n_clusters, "silhouette": sil, "key": key})
    return pd.DataFrame(rows)


def silhouette_sweep_kmeans(adata, ks, random_state=42):
    rows = []
    X_pca = adata.obsm["X_pca"]
    for k in ks:
        km = KMeans(n_clusters=k, random_state=random_state, n_init=10).fit(X_pca)
        sil = silhouette_score(X_pca, km.labels_)
        adata.obs[f"kmeans_k{k}"] = pd.Categorical(km.labels_.astype(str))
        rows.append({"k": k, "silhouette": sil})
    return pd.DataFrame(rows)


def label_agreement(adata, cluster_key):
    """Compare a clustering against the true condition labels."""
    truth = adata.obs["condition"].astype(str).values
    pred = adata.obs[cluster_key].astype(str).values
    ari = adjusted_rand_score(truth, pred)
    nmi = normalized_mutual_info_score(truth, pred)
    contingency = pd.crosstab(adata.obs["condition"], adata.obs[cluster_key])
    return ari, nmi, contingency

/var/folders/nr/mb9t16753gl29cqg5ry9kft00000gn/T/ipykernel_70668/3463357260.py:6: FutureWarning: Use `scanpy.set_figure_params` instead
  sc.settings.set_figure_params(dpi=80, facecolor="white")


In [ ]:
from src.utils import load_smartseq, to_anndata

CELL_LINES = ["MCF7", "HCC1806"]

ImportError: cannot import name 'to_anndata' from 'src.utils' (/Users/gift/Projects/Python_projects/AI_Lab/src/utils.py)

## 1 — Build AnnData for both cell lines

In [ ]:
adatas = {}
for cl in CELL_LINES:
    X, y = load_smartseq(cl, split="train")
    adatas[cl] = to_anndata(X, y, dataset=f"{cl}_SmartSeq")
    print(f"{cl}: {adatas[cl]}")

## 2 — Run the scanpy pipeline (PCA, neighbours, UMAP)

In [ ]:
for cl, a in adatas.items():
    run_scanpy_pipeline(a, n_pcs=30, n_neighbors=15)

In [ ]:
# Quick PCA elbow to justify the n_pcs choice.
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, (cl, a) in zip(axes, adatas.items()):
    var = a.uns["pca"]["variance_ratio"]
    ax.plot(np.arange(1, len(var) + 1), np.cumsum(var) * 100, "o-")
    ax.axhline(80, ls="--", c="grey", lw=0.8)
    ax.set_title(f"{cl} — cumulative variance explained")
    ax.set_xlabel("# PCs")
    ax.set_ylabel("% variance")
plt.tight_layout()
plt.show()

## 3 — UMAP coloured by the (held-out) condition

In [ ]:
for cl, a in adatas.items():
    sc.pl.umap(a, color="condition", title=f"{cl} UMAP — condition", frameon=False, show=True)

If Hypo and Norm cells separate visually, the **dominant axis of variation in
the data is hypoxia** — encouraging for the supervised step. If they overlap
heavily, hypoxia may share its signal with other technical or biological
factors (cell cycle, batch, plate position for SmartSeq).

## 4 — Silhouette sweep: Leiden resolution

In [ ]:
leiden_results = {}
for cl, a in adatas.items():
    df = silhouette_sweep_leiden(a, resolutions=[0.1, 0.2, 0.3, 0.5, 0.8, 1.0, 1.5])
    leiden_results[cl] = df
    print(f"\n{cl} Leiden sweep:")
    print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, (cl, df) in zip(axes, leiden_results.items()):
    ax.plot(df["resolution"], df["silhouette"], "o-")
    for _, row in df.iterrows():
        ax.annotate(
            f"k={int(row['n_clusters'])}",
            (row["resolution"], row["silhouette"]),
            textcoords="offset points",
            xytext=(0, 6),
            fontsize=8,
            ha="center",
        )
    ax.set_title(f"{cl} — silhouette vs Leiden resolution")
    ax.set_xlabel("resolution")
    ax.set_ylabel("silhouette (PCA space)")
plt.tight_layout()
plt.show()

## 5 — Silhouette sweep: KMeans `k`

In [ ]:
kmeans_results = {}
for cl, a in adatas.items():
    df = silhouette_sweep_kmeans(a, ks=range(2, 7))
    kmeans_results[cl] = df
    print(f"\n{cl} KMeans sweep:")
    print(df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, (cl, df) in zip(axes, kmeans_results.items()):
    ax.plot(df["k"], df["silhouette"], "o-")
    ax.set_title(f"{cl} — silhouette vs k (KMeans)")
    ax.set_xlabel("k")
    ax.set_ylabel("silhouette (PCA space)")
plt.tight_layout()
plt.show()

## 6 — Pick the best clustering and compare to ground truth

In [ ]:
# Best Leiden by silhouette (per cell line).
results = []
for cl, a in adatas.items():
    best = leiden_results[cl].loc[leiden_results[cl]["silhouette"].idxmax()]
    best_kmeans = kmeans_results[cl].loc[kmeans_results[cl]["silhouette"].idxmax()]
    ari_l, nmi_l, cont_l = label_agreement(a, best["key"])
    ari_k, nmi_k, cont_k = label_agreement(a, f"kmeans_k{int(best_kmeans['k'])}")
    print(f"\n=== {cl} ===")
    print(
        f"Best Leiden: res={best['resolution']:.2f}  k={int(best['n_clusters'])}  "
        f"silhouette={best['silhouette']:.3f}  ARI={ari_l:.3f}  NMI={nmi_l:.3f}"
    )
    print(cont_l)
    print(
        f"\nBest KMeans: k={int(best_kmeans['k'])}  silhouette={best_kmeans['silhouette']:.3f}  "
        f"ARI={ari_k:.3f}  NMI={nmi_k:.3f}"
    )
    print(cont_k)
    results.append(
        {
            "cell_line": cl,
            "leiden_res": best["resolution"],
            "leiden_k": int(best["n_clusters"]),
            "leiden_silhouette": best["silhouette"],
            "leiden_ari": ari_l,
            "leiden_nmi": nmi_l,
            "kmeans_k": int(best_kmeans["k"]),
            "kmeans_silhouette": best_kmeans["silhouette"],
            "kmeans_ari": ari_k,
            "kmeans_nmi": nmi_k,
        }
    )
pd.DataFrame(results)

## 7 — UMAP coloured by best clustering

In [ ]:
for cl, a in adatas.items():
    best_key = leiden_results[cl].loc[leiden_results[cl]["silhouette"].idxmax(), "key"]
    sc.pl.umap(
        a, color=[best_key, "condition"], title=[f"{cl} — {best_key}", f"{cl} — condition"], frameon=False, show=True
    )

## 8 — Discussion

Things to write up in the report based on what the numbers say above:

- Did the *best* silhouette correspond to **2 clusters** (matching Hypo/Norm),
  or did the data prefer a different number? On SmartSeq, the answer is often
  **not 2** — the dominant axis can be cell cycle or batch effects, not
  hypoxia.
- How does **silhouette align with ARI/NMI vs the true labels**? A high
  silhouette + low ARI means the data has strong structure that **isn't**
  hypoxia. A low silhouette + high ARI means hypoxia is weak but real.
- Compare KMeans (geometric, k forced) with Leiden (graph-based, k
  discovered). They often disagree on the optimal partition.